# DroneHacks flight notebook

Commands are instructions for your companion terminal; no kernel is attached in the lab. Flight commands require an explicit operator action. Completion is self-reported, not flight qualification.


01 / MEET YOUR DRONE

Start in the companion.


        The terminal is an ordinary Linux shell inside your companion computer. The drone is a simulated vehicle controlled by actual PX4 firmware running in SITL.


        Two computers, two jobs

A real drone usually has a flight controller and a companion computer. PX4 runs on the flight controller’s microcontroller and handles stabilization, motors, flight modes and failsafes. Your mission and vision software run on a Linux companion, such as a Jetson. Here, PX4 runs as software-in-the-loop (SITL), while your companion runs in a real Linux container.

ROS 2 lets programs exchange typed messages on topics: a camera publishes images; your estimator subscribes. uXRCE-DDS connects the resource-constrained PX4 client to an agent, which makes selected PX4 data available to ROS 2 and carries supported commands back. The agent and PX4 mapping are already configured in this lab. The supplied student drone setup is intended to provide this bridge too; verify its exact image and message versions before using your code on hardware.


        What is running in Docker?

A process is a running program. A Docker container packages Linux programs and their dependencies with a defined filesystem and access boundary; it is not another flight controller. Your editor, terminal and editable onboard programs share the companion’s user environment. PX4, virtual sensor generation, transport and independent grading are managed outside your editable application.

You do not need to run Docker inside this terminal. Open onboard/config/services.json to see the actual commands that launch the onboard processes, and onboard/README.md for their lifecycle. Saving source does not restart a process. Installed packages last for this GPU session; save dependency requirements with your project.


        Launch a field session

Open Session, choose Outdoor field · GNSS, and launch. Wait for READY. Return to Workbench and open the editor’s terminal with Ctrl + `. Your editor and terminal stay mounted when you return to Learn.

The maze is a later, GPS-denied challenge. Begin outdoors so you can learn the interfaces before depending on vision for position.


        Inspect the real environment

Copy this into that terminal. It reads your environment and session state; it does not arm the vehicle.


          BASH · READ ONLY
```bash
cd /var/lib/flyx-lab/workspace
pwd
uname -m
printenv ROS_DISTRO ROS_DOMAIN_ID
ps -ef
```



          Check: the session is Outdoor field · GNSS with your chosen drone, the runtime is READY, and the vehicle reports disarmed and landed. A missing or stale report means “unknown”, not ready.


        
        Discover, then read actual content

List the ROS graph. Seeing a topic name is only discovery: your own subscription can create that name. The next chapter reads timestamps, validity flags and values for 30 seconds.


          BASH · READ ONLY
```bash
ros2 topic list -t
ros2 interface show px4_msgs/msg/VehicleLocalPosition
```



          Allow at least 25 seconds for DDS discovery before calling a stream absent. The exercises use a 30-second wall-time window. A timeout exit code of 124 simply means that window ended.


        
        What transfers to a drone?Your Python, ROS messages and Offboard control boundary are real. This companion is x86 Ubuntu/Jazzy, not an ARM Jetson or a JetPack performance emulator. Your editable mission and onboard estimation run inside it. PX4, virtual sensor generation and the simulator bridge stay managed. Hardware needs its own reviewed launch, drivers, calibration, message versions and clock settings.


### My notes

No notes yet.


02 / READ PX4 TELEMETRY

Talk to the autopilot.


        PX4’s uXRCE-DDS client connects to the managed agent. These ROS topics come from the running SITL autopilot, through its actual uORB/DDS mapping.


        Position, validity and height

PX4 local position uses NED: x north, y east, z down, in metres. A negative z change means climbing. These are estimator outputs, not simulator truth.


          BASH · 30 SECOND OBSERVATION
```bash
timeout 30s ros2 topic echo /fmu/out/vehicle_local_position_v1 px4_msgs/msg/VehicleLocalPosition --qos-reliability best_effort --qos-durability volatile
```



          Check: timestamps advance; xy_valid and z_valid are true before position control. Keep dist_bottom_valid beside dist_bottom: a plausible number alone does not establish a valid ground distance.


        
        Arming state and flight mode

This second subscription reads PX4’s own vehicle state. The topic has a _v1 suffix while its message type remains VehicleStatus; topic and message versions must match the installed firmware and px4_msgs pin.


          BASH · READ ONLY
```bash
timeout 30s ros2 topic echo /fmu/out/vehicle_status_v1 px4_msgs/msg/VehicleStatus --qos-reliability best_effort --qos-durability volatile
ros2 interface show px4_msgs/msg/VehicleStatus
```



          The interface definition gives the enum values for arming_state and nav_state. The subscriptions explicitly use compatible best-effort QoS. They do not publish flight commands.


        
        A range reading is not a parameter

The downward rangefinder casts a simulated beam and sends MAVLink DISTANCE_SENSOR into PX4. PX4’s estimator uses it according to the scenario’s parameter file. EKF2_RNG_CTRL is a fusion parameter; dist_bottom is an estimated distance to the bottom surface.

The native interface exports /fmu/out/distance_sensor, whose current_distance is the beam measurement in metres. It is a slant range, not automatically vertical height. /fmu/in/distance_sensor injects a measurement into PX4; it is not a sensor output. This lesson changes no FC parameters.


          BASH · READ ONLY
```bash
timeout 30s ros2 topic echo /fmu/out/distance_sensor px4_msgs/msg/DistanceSensor --qos-reliability best_effort --qos-durability volatile
```



        
        Find the real control boundary

These message definitions are the inputs your editable refnav/node.py publishes. Reading the schema is safe; sending a second competing setpoint stream is a separate control action.


          BASH · MESSAGE SCHEMAS
```bash
ros2 interface show px4_msgs/msg/OffboardControlMode
ros2 interface show px4_msgs/msg/TrajectorySetpoint
ros2 interface show px4_msgs/msg/VehicleCommand
```



          /fmu/in/offboard_control_mode declares the active control interface.

/fmu/in/trajectory_setpoint carries NED position targets.

/fmu/in/vehicle_command carries commands such as arm or land.


          In this starter, Python updates position setpoints at 20 Hz. PX4’s internal controllers and the simulator physics have their own rates. A GPU rating is not evidence of a 250 Hz Python vision loop.


        
        Primary reference: PX4 ROS 2 user guide. These commands follow this lab’s pinned interfaces, not an assumed upstream topic list.


### My notes

No notes yet.


03 / READ THE CAMERAS

Images your code can use.


        Your node consumes native sensor_msgs/Image and CameraInfo messages. The simulation supplies them where a hardware camera driver would normally publish.


        USB and ROS are different interfaces.On hardware, lsusb inspects USB devices; SBUS is an RC interface. An OAK camera is normally opened by DepthAI, which runs its device pipeline and publishes data through a driver. This sim does not emulate OAK USB/XLink, device firmware, or a device that depthai.Device() can open. We do not add a fake entry to lsusb. This walkthrough tests your ROS camera-processing code, not the hardware driver.


        One onboard camera interface

The field and maze expose the same native camera names. This selection shows the world’s context; it does not switch your simulation. Check the Session world and read fresh PX4 state after every world switch.


          Topic examples forIndoor maze · visionOutdoor field · GNSSSky Beta · Indoor maze · visionSky Beta · DroneHacks course · visionSky Beta · DroneHacks prepared course · visionSky Beta · Outdoor field · GNSS
          Same native camera names in both worlds. Sky Beta · DroneHacks prepared course · vision: read actual content to establish liveness.


          BASH · SET TOPIC VARIABLES
```bash
export LEFT_TOPIC=/stereo/left/image
export RIGHT_TOPIC=/stereo/right/image
export DEPTH_TOPIC=/camera/depth/image_raw
export DEPTH_INFO_TOPIC=/camera/depth/camera_info
```



          StreamRoleDeclared ROS topic

Main RGB · renderedRenderedBrowser preview only

Stereo leftModeled sensor/stereo/left/image

Stereo rightModeled sensor/stereo/right/image

Stereo depthModeled sensor/camera/depth/image_raw

Follow · rearObserver onlyBrowser preview only


          Left/right/depth use the configured sensor-realism pipeline. Main RGB is a render, without that stereo noise model. Follow is an external observer. Those two views remain browser previews and are excluded from the onboard ROS interface. Your node reads native sensor messages.


        
        Inspect calibration and capture time

Use the same terminal where you set the variables. This prints calibration, not image byte arrays. The frame ID identifies the optical frame; use CameraInfo and TF when turning pixels into directions. Do not assume RGB and depth are already aligned.


          BASH · READ ONLY
```bash
timeout 30s ros2 topic echo "$DEPTH_INFO_TOPIC" sensor_msgs/msg/CameraInfo --qos-reliability best_effort --qos-durability volatile
```



          Image headers use the simulation capture clock. Processing and receipt happen later; never subtract a wall-clock timestamp from a simulation timestamp and call it latency. On hardware, configure the device/ROS clock relationship explicitly.


        
        Write a real Python subscriber

Copy this block once to create camera_probe.py in your workspace. It refuses to overwrite an existing file. Then open that file in the editor: the entire subscriber is yours to read and change.


          BASH · CREATE camera_probe.py
```bash
cd /var/lib/flyx-lab/workspace
(
set -o noclobber
cat > camera_probe.py <<'FLYX_TUTORIAL_PY'
"""Read a real ROS Image subscription; no flight commands or simulator truth.

python3 camera_probe.py /your/camera/image_raw --sim-time --depth --seconds 30
Remove --sim-time on hardware. --depth interprets 16UC1 as mm and 32FC1 as m;
leave it off for left/right/RGB. No USB/DepthAI device is emulated by this node.
"""
import argparse
import json
import math
import time

import numpy as np


def describe(message, *, depth=False):
    """Honor encoding, byte order and padded rows; never reinterpret RGB as depth."""
    h, w, step = message.height, message.width, message.step
    if not 0 < h <= 4096 or not 0 < w <= 4096 or step > 65536 or len(message.data) != step * h:
        raise ValueError('invalid image dimensions or buffer length')
    pixel_bytes = {'mono8': 1, 'mono16': 2, 'rgb8': 3, 'bgr8': 3, 'rgba8': 4,
                   'bgra8': 4, '16UC1': 2, '32FC1': 4}.get(message.encoding)
    if pixel_bytes is None or step < w * pixel_bytes:
        raise ValueError('unsupported encoding or image row shorter than its declared width')
    out = dict(encoding=message.encoding, width=w, height=h, step=step,
               frame_id=message.header.frame_id,
               capture_s=message.header.stamp.sec + message.header.stamp.nanosec * 1e-9)
    if depth:
        dtype = {'16UC1': 'u2', '32FC1': 'f4'}.get(message.encoding)
        if dtype is None:
            raise ValueError('depth requires 16UC1 millimetres or 32FC1 metres')
        dt = np.dtype(('>' if message.is_bigendian else '<') + dtype)
        if step < w * dt.itemsize:
            raise ValueError('image row shorter than its declared width')
        image = np.ndarray((h, w), dtype=dt, buffer=bytes(message.data), strides=(step, dt.itemsize))
        metres = image.astype(np.float32) * (0.001 if dtype == 'u2' else 1.0)
        valid = np.isfinite(metres) & (metres > 0)
        out.update(valid_fraction=float(valid.mean()),
                   median_m=float(np.median(metres[valid])) if valid.any() else None)
    return out


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('topic')
    parser.add_argument('--depth', action='store_true')
    parser.add_argument('--sim-time', action='store_true')
    parser.add_argument('--seconds', type=float, default=30)
    args = parser.parse_args(argv)
    if not math.isfinite(args.seconds) or not 25 <= args.seconds <= 300:
        parser.error('--seconds must be 25..300 wall seconds (allow DDS discovery)')
    import rclpy
    from rclpy.node import Node
    from rclpy.parameter import Parameter
    from rclpy.qos import qos_profile_sensor_data
    from sensor_msgs.msg import Image

    rclpy.init()
    node = Node('camera_lesson')
    node.set_parameters([Parameter('use_sim_time', value=args.sim_time)])
    state = dict(messages=0, advancing=0, invalid=0, repeated_or_reordered=0)
    latest = None
    last_stamp = None
    last_advance = None

    def receive(message):
        nonlocal latest, last_stamp, last_advance
        state['messages'] += 1
        try:
            summary = describe(message, depth=args.depth)
        except (ValueError, TypeError) as error:
            state['invalid'] += 1
            node.get_logger().warning(str(error))
            return
        if last_stamp is not None and summary['capture_s'] <= last_stamp:
            state['repeated_or_reordered'] += 1
            return
        last_stamp, last_advance, latest = summary['capture_s'], time.monotonic(), summary
        state['advancing'] += 1

    subscription = node.create_subscription(Image, args.topic, receive, qos_profile_sensor_data)
    start = time.monotonic()
    next_report = start + 1
    try:
        while rclpy.ok() and time.monotonic() - start < args.seconds:
            rclpy.spin_once(node, timeout_sec=0.2)
            now = time.monotonic()
            if now >= next_report:
                print(json.dumps(dict(state, last_image=latest), allow_nan=False), flush=True)
                next_report = now + 1
        age = None if last_advance is None else time.monotonic() - last_advance
        ok = state['advancing'] >= 2 and state['invalid'] == 0 and age is not None and age <= 2
        print(json.dumps(dict(state, fresh_advancing_images=ok,
                             receipt_age_wall_s=age, observation_wall_s=time.monotonic() - start)), flush=True)
        return 0 if ok else 1
    finally:
        node.destroy_subscription(subscription)
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    raise SystemExit(main())
FLYX_TUTORIAL_PY
)
```



          create_subscription receives real images with sensor-data QoS. The callback reads dimensions, capture stamps and frame IDs. With --depth, it respects row padding and byte order, converts millimetres to metres, and excludes zero/invalid pixels. It reports whole-frame depth statistics, not collision-free clearance.


        
        Run it against the live cameras

Each command observes the actual ROS stream for 30 wall seconds and exits. It has no flight-command publisher. A single initial sample is not enough: the final report checks advancing capture stamps and recent receipts. Exit 1 means that check did not pass.


          BASH · LIVE SENSOR READERS
```bash
python3 camera_probe.py "$LEFT_TOPIC" --sim-time --seconds 30
python3 camera_probe.py "$RIGHT_TOPIC" --sim-time --seconds 30
python3 camera_probe.py "$DEPTH_TOPIC" --sim-time --depth --seconds 30
```



          Check: advancing increases; fresh_advancing_images ends true; depth units are metres. Frames may be missed by a best-effort observer. These checks do not certify complete capture or camera latency. Do not add ROS-topic latency as if it were measured USB transport latency.


          For an exercise, add a centre-region depth statistic inside describe(), run again, and compare it while the drone moves in the next chapter. Missing depth must remain “unknown”, never “clear”.


        
        Inspect the onboard estimator you own

The maze runs the actual onboard/slam/cuvslam_lib_node.py and onboard/vio/ev_bridge.py from this workspace. The field uses your PX4 health and odometry adapters. The NVIDIA library is an installed dependency. These commands inspect real processes and launch arguments; they never arm.


          BASH · READ YOUR SERVICES
```bash
cd /var/lib/flyx-lab/workspace
python3 onboard/services.py status
ps -ef
cat onboard/config/services.json
cat onboard/README.md
```



          Edit the source or YAML, then explicitly stop/start these services while disarmed to apply it. A save does not restart a running Python process. Stopped or failed vision has no hidden fallback. Ordinary ROS scripts run directly in your terminal; the optional Run button adds source snapshots, recording and independent grading.


        
        Hardware reference: Luxonis DepthAI ROS driver. Hardware topics and extrinsics need a reviewed mapping to your node’s inputs.


### My notes

No notes yet.


04 / FOLLOW CUVSLAM

How pictures become position.


        Use the working estimator as something you can understand and change. It is a starting point for GPS-denied flight, not a complete navigation algorithm.


        Check the installed source version first.This file walkthrough describes the portable package, flyx-onboard-source/2. Read onboard-source.json and onboard/config/services.json in your workspace. Earlier /1 releases use the same main file paths but include extra timing helpers, different callback names and no standard processing-status topic. Publishing this notebook does not upgrade your worker or saved source. Follow the files actually installed; do not start a second estimator.


        Start with the data path

The maze’s onboard process receives left and right monochrome images, each camera’s calibration, the camera IMU and TF transforms. It pairs stereo images by capture time, sends the pair and eligible IMU measurements to NVIDIA cuVSLAM, then publishes an estimated pose. A separate bridge checks and converts that estimate for PX4. It never substitutes simulator ground truth.

Stereo + IMU→cuVSLAM→Vision bridge→PX4Depth is available for your obstacle detection or planning code; this wrapper’s tracking call uses stereo images and the configured IMU. It does not feed the depth image into that call. An odometry estimate says where the vehicle thinks it is; it does not choose a safe route or avoid a wall.


        Read these files in this order

Open in your editorWhat to look for

onboard/config/services.jsonThe vio process list and actual Python launch commands. The field uses the gnss profile instead.

onboard/config/vio.yamlCamera/IMU frames, IMU noise, minimum visual observations and IMU-only policy. Read the effective launch configuration too; deployment overrides matter.

onboard/slam/cuvslam_lib_node.py_try_init builds calibration and transforms. _image pairs capture stamps; _track_frame registers IMU and calls tracker.track; _publish produces odometry.

onboard/vio/ev_bridge.py_on_odom checks time, validity, health and sole-publisher ownership before publishing VehicleOdometry.

onboard/vio/frames.pyOptical/body/world conventions and conversion to PX4. NED requires a declared heading reference; an unreferenced estimate uses FRD.

The NVIDIA tracker library is installed in the companion. These Python wrappers are visible, editable application source. On your supplied student drone, cuVSLAM is the intended preinstalled starter estimator; confirm that hardware’s package, camera driver and calibration. It is reasonable to replace or extend it for your final solution.


        Inspect without changing the vehicle

BASH · READ ONLY
```bash
cd /var/lib/flyx-lab/workspace
python3 onboard/services.py status
cat onboard-source.json
cat onboard/config/services.json
sed -n '1,180p' onboard/slam/cuvslam_lib_node.py
sed -n '1,140p' onboard/vio/ev_bridge.py
```


In a maze session with the estimator running, inspect its output for 30 seconds. These commands only subscribe; they do not start a second tracker or publish control messages.

BASH · MAZE ESTIMATOR OBSERVATION
```bash
timeout 30s ros2 topic echo /visual_slam/tracking/odometry nav_msgs/msg/Odometry --qos-reliability best_effort --qos-durability volatile
timeout 30s ros2 topic echo /estimator/healthy std_msgs/msg/Bool --qos-reliability best_effort --qos-durability volatile
```


Check: advancing capture stamps, the actual processing result, sensible frame IDs and covariance, and current health. A returned pose alone is not proof of visual tracking: an IMU-only estimate can still exist when visual features are lost.


        A useful first exerciseSketch the pipeline in your notes. Explain where you would reject an old image, where the world-to-PX4 conversion happens, and which part must handle tracking loss. Before editing/restarting onboard services, finish the mission and confirm disarmed/landed. Never run two vision publishers into PX4. Keep failed evidence and the health gates when experimenting.


### My notes

No notes yet.


05 / FLY, EDIT, FLY AGAIN

Change code. See it fly.


        Run a real Offboard mission against the PX4 SITL vehicle. Then edit your own source, reset and observe a different climb.


        This lesson uses GPS. The maze challenge has NO GPS.Choose Outdoor field · GNSS in Session and wait for READY. The example waits for advancing, usable native GPS measurements and refuses their absence. The challenge needs its configured vision estimator; a simulated GPS field pass does not validate navigation inside the maze.


        Verify the field and open the actual controller

Use Session to check the outdoor field (field-gnss or sky-beta-field-gnss), READY, disarmed, landed and no active program. Read GPS fix/accuracy data below. If readiness is false, inspect its reasons rather than removing a gate.


          BASH · READ ONLY
```bash
timeout 30s ros2 topic echo /fmu/out/vehicle_gps_position px4_msgs/msg/SensorGps --qos-reliability best_effort --qos-durability volatile
```



          GPS fix and accuracy fields describe the receiver report, not guaranteed position truth. Open FLIGHT_CODE.md and refnav/core.py. The climb phase reduces NED z; refnav/node.py publishes those targets to PX4.


          BASH · CHECK LOCAL SOURCE
```bash
cd /var/lib/flyx-lab/workspace
python3 -c 'from refnav import core; print(core.__file__)'
python3 -m pytest -q tests
```



          The import must point into this workspace. The tests exercise your local decisions against a kinematic fake; they do not start a drone or prove real flight dynamics. If pytest is absent, install it into your own development environment and record that dependency.


        
        Create the GPS lesson entrypoint

This creates gps_takeoff.py beside your editable refnav/. It selects 1.2 m height and a five-second hover, checks the GPS profile, then starts that local flight package. The full decision logic remains visible in your project.


          BASH · CREATE gps_takeoff.py
```bash
cd /var/lib/flyx-lab/workspace
(
set -o noclobber
cat > gps_takeoff.py <<'FLYX_TUTORIAL_PY'
"""GPS field lesson. Run with `python3 gps_takeoff.py` from your lab workspace.

The editable refnav/ package beside this file owns every mission decision and
ROS command. This example checks actual native GPS messages on hardware or SITL
and refuses the GPS-denied challenge without publishing commands.
"""
from importlib.util import find_spec
from pathlib import Path

HEIGHT_M = 1.2
HOVER_S = 5.0

PROJECT = Path(__file__).resolve().parent
package = find_spec('refnav')
if package is None or package.origin is None or Path(package.origin).resolve().parent != PROJECT / 'refnav':
    raise SystemExit('The full editable refnav/ package must be beside this file. See FLIGHT_CODE.md.')

from refnav import node  # noqa: E402
from refnav.gps import wait_for_gps  # noqa: E402
from refnav.launch import arguments  # noqa: E402

if __name__ == '__main__':
    import sys
    if not wait_for_gps():
        raise SystemExit('GPS lesson requires Outdoor field · GNSS. The maze challenge has NO GPS.')
    print(f'GPS lesson · local flight code: {Path(node.__file__).parent}', flush=True)
    node.main(arguments(PROJECT, height_m=HEIGHT_M, hover_s=HOVER_S, ros_args=sys.argv[1:]))
FLYX_TUTORIAL_PY
)
```



        
        Run and watch the SITL drone

The next command can arm and fly the simulated aircraft. Run it only in your isolated lab session, with no competing control program. Saving a file does not fly it; this explicit Run action does.


          BASH · STARTS SIMULATED FLIGHT
```bash
python3 gps_takeoff.py
```



          In the workbench, choose Follow to watch takeoff and Depth to see the onboard view. Follow is for you, not an input to the mission. Watch Run output: readiness → Offboard/arm → climb → hover → PX4 landing → landed/disarmed. Your flight may abort on a real readiness or sensor issue; keep its evidence.


          Python runs the saved source directly in this companion; the output appears here in the notebook or terminal. The same command runs on a configured Jetson. Exit success alone is not a flight grade. If you need a ROS recording, start ros2 bag record for the topics you need before the mission; direct Python does not create a managed Run archive. The optional workbench Run button adds a managed snapshot and recording.


        
        Edit, reset and run a second time

After the vehicle has landed/disarmed and the program has finished, edit HEIGHT_M in gps_takeoff.py from 1.2 to 1.5 and save. Reset the world and wait for fresh READY before running again. The file persists; the simulated vehicle restarts.


          SIMULATION ONLY
```bash
# In Session, choose Reset simulation, then wait for READY.
# This resets the vehicle; your project files remain.
```



          Once Session and fresh PX4 telemetry confirm READY/disarmed/landed with no active run, use python3 gps_takeoff.py again. Compare the hover level in Follow and the z change on PX4 local position. Direct Python loads your local package. Save before launch and finish the mission before editing flight code.


          Next, change Params.climb_rate_mps in refnav/core.py from its baseline 0.5 to 0.3, rerun the unit tests, reset and repeat. The slower ramp should take longer to reach the same target. Keep health, takeover and landing safeguards while changing behavior.


        
        Recovery and evidenceUse the workbench’s Land or Stop program controls for an active mission; those request the managed recovery path for workbench-managed runs. A directly launched Python program is a separate process: its own PX4 landing/takeover behavior must handle recovery, and interrupting Python does not prove touchdown. Neither a delivered request nor closing the browser proves touchdown. Observe PX4’s actual landed/disarmed state. Keep failed-run recordings; do not reset to erase a failure before collecting its cause.

Download project snapshots and recordings under Runs & recordings. Reset preserves project files and packages; Release saves the project/recordings but does not preserve session-installed packages. Hardware deployment needs separate qualification.


### My notes

No notes yet.


06 / TAKE IT TO HARDWARE

One explicit start. An autonomous mission.


        Keep the application’s ROS interfaces. Replace the simulator’s sensor and transport setup with the reviewed hardware deployment.


        Connect before the mission

In the real drone project, configure the aircraft’s Wi-Fi hotspot and connect your laptop to it. Use your reviewed local software or SSH to inspect health and select the mission. You must set up the hotspot and access method for that aircraft; this website does not create a hardware Wi-Fi link.

Confirm the flight-controller firmware and px4_msgs versions, uXRCE-DDS client/agent transport, actual OAK camera driver, calibration, TF, clock selection and estimator health. Hardware normally uses ROS system time, not the simulator’s /clock. Hardware parameters require their own review; never copy a simulation parameter file blindly.


        Make starting a deliberate action

After the operator’s checks, a software button or an explicitly configured RC switch can request mission start. We recommend a clear software start with visible readiness and acknowledgement. Simply booting the drone, connecting to Wi-Fi or opening a page must not arm or launch it.

The pilot’s RC remains connected and available for takeover and the configured emergency/kill action. Verify those actions during hardware qualification. After the explicit mission start, the project goal is autonomous takeoff, navigation and landing at the final destination, without routine control inputs. The software must still yield to takeover and react safely to lost estimation or connectivity.


        Transfer code, then qualify the deployment

Keep mission logic, estimator logic and message contracts together with dependency pins and configuration. Change the camera/transport launch and platform-specific dependencies for the actual Jetson and flight controller. The sim companion is x86; a Jetson is ARM. Using the same ROS interface does not make USB drivers, binaries, exposure, clocks or performance identical.

A simulation pass supports your development. It does not qualify an aircraft. Progress through read-only sensor checks, disarmed bench integration, reviewed safety tests and controlled flights. Compare real camera timing, calibration, tracking loss and vehicle response with your simulation evidence before increasing the mission’s scope.


        Your mission, in one sentenceWrite the start condition, intended route, final landing condition, and what happens on vision loss or pilot takeover. If any of those is unclear, resolve it before a hardware flight.


### My notes

No notes yet.
